In [2]:

# ============================================================
# CS 340 Project Two - Grazioso Salvare Dashboard
# Developed by Rosalie Reblora
# ============================================================

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

JupyterDash.infer_jupyter_proxy_config()


#### FIX ME #####
# Change animal_shelter to match your CRUD module.
# ORIGINAL: from animal_shelter import AnimalShelter

# FIXED: Import the CRUD module from Project One.
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

#### FIX ME #####
# Update username and password.

# FIXED: Use your MongoDB aacuser account.
username = "aacuser"
password = "my_password"

# Connect to MongoDB through the CRUD module.
db = AnimalShelter(username, password)

# Retrieve all records from MongoDB.
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectId because Dash cannot serialize it.
df.drop(columns=["_id"], inplace=True, errors="ignore")

# Verify database connection.
if df.empty:
    raise RuntimeError(
        "No records returned. Check MongoDB credentials."
    )


#### FIX ME #####
# Add MongoDB queries for the three rescue categories.

# FIXED: Define the rescue filters using MongoDB queries.
def get_rescue_query(rescue_type):

    # Water Rescue
    if rescue_type == "Water Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Mountain or Wilderness Rescue
    elif rescue_type == "Mountain or Wilderness Rescue":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # Disaster or Individual Tracking
    elif rescue_type == "Disaster or Individual Tracking":
        return {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # Reset - Return all animals.
    return {}


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)


#### FIX ME #####
# Add Grazioso Salvare's logo.

# FIXED: Load the logo using Base64.
# Change this filename to match your actual Codio image.
image_filename = "Grazioso Salvare Logo.png"

with open(image_filename, "rb") as image_file:
    encoded_image = base64.b64encode(
        image_file.read()
    ).decode("utf-8")


#### FIX ME #####
# Place the HTML image in the dashboard layout.
# Include your unique identifier.

app.layout = html.Div([

    # FIXED: Add logo linked to SNHU and your name.
    html.Center([

        html.A(
            html.Img(
                src="data:image/png;base64," + encoded_image,
                style={"width": "180px"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.H1("Grazioso Salvare Rescue Dashboard"),
        html.H3("Developed by Rosalie Reblora")
    ]),

    html.Hr(),


    #### FIX ME #####
    # Add interactive filtering options.

    # FIXED: Radio buttons for rescue categories.
    html.Div([

        html.H3("Select Rescue Training Category"),

        dcc.RadioItems(
            id="filter-type",

            options=[
                {
                    "label": "Reset / All Animals",
                    "value": "Reset"
                },
                {
                    "label": "Water Rescue",
                    "value": "Water Rescue"
                },
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "Mountain or Wilderness Rescue"
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "Disaster or Individual Tracking"
                }
            ],

            value="Reset",

            labelStyle={
                "display": "inline-block",
                "marginRight": "20px"
            }
        )
    ]),

    html.Hr(),


    # Interactive animal data table
    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),


        #### FIX ME #####
        # Set up user-friendly interactive DataTable features.

        # FIXED: Features from your Module Six Milestone.
        row_selectable="single",
        selected_rows=[0],

        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        },

        style_header={
            "backgroundColor": "#E8EDF2",
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),


    # Display the pie chart and map side by side.
    html.Div(
        className="row",

        style={
            "display": "flex",
            "flexWrap": "wrap",
            "gap": "20px"
        },

        children=[

            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"flex": "1 1 350px"}
            ),

            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"flex": "1 1 450px"}
            )
        ]
    )

], style={"padding": "20px"})


#############################################
# Interaction Between Components / Controller
#############################################


#### FIX ME #####
# Add code to filter the interactive data table
# using MongoDB queries.

# FIXED: Query MongoDB whenever a rescue filter changes.
@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "selected_rows"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):

    query = get_rescue_query(filter_type)

    # Use the CRUD read method from Project One.
    results = db.read(query)

    filtered_df = pd.DataFrame.from_records(results)

    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    data = filtered_df.to_dict("records")

    # Select the first row if records are available.
    selected = [0] if data else []

    return data, selected


#### FIX ME #####
# Add a second chart of your choice.

# FIXED: Breed distribution pie chart.
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(viewData):

    if viewData is None:
        viewData = df.to_dict("records")

    chart_df = pd.DataFrame.from_records(viewData)

    if chart_df.empty or "breed" not in chart_df.columns:
        return html.P("No animals match this rescue category.")

    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ["breed", "count"]

    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Animal Breed Distribution"
    )

    return [
        dcc.Graph(figure=fig)
    ]


# Highlight selected columns.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):

    # FIXED: Prevent NoneType errors.
    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {"column_id": i},
            "backgroundColor": "#D2F3FF"
        }
        for i in selected_columns
    ]


#### FIX ME #####
# Update geolocation chart based on the selected animal.

# FIXED: Map updates when the selected row or
# filtered table data changes.
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows")
)
def update_map(viewData, index):

    if viewData is None:
        viewData = df.to_dict("records")

    dff = pd.DataFrame.from_records(viewData)

    if dff.empty:
        return html.P("No location available.")

    # Default to the first animal.
    row = 0

    if index and 0 <= index[0] < len(dff):
        row = index[0]

    animal = dff.iloc[row]

    latitude = pd.to_numeric(
        animal.get("location_lat"),
        errors="coerce"
    )

    longitude = pd.to_numeric(
        animal.get("location_long"),
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P(
            "No valid location coordinates for this animal."
        )

    breed = str(animal.get("breed", "Unknown"))
    name = str(animal.get("name", "Unnamed"))

    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "450px"
            },

            center=[latitude, longitude],
            zoom=10,

            children=[
                dl.TileLayer(),

                dl.Marker(
                    position=[latitude, longitude],

                    children=[
                        dl.Tooltip(breed),

                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(name),
                            html.P("Breed: " + breed)
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard in JupyterLab.
app.run_server(mode="inline")
